In [87]:
!pip install torch torchaudio transformers faiss-cpu librosa tqdm

In [88]:
import os
import requests
from tqdm import tqdm

In [89]:
AUDIO_URLS = [
    "https://www.mmsp.ece.mcgill.ca/Documents/AudioFormats/WAVE/Samples/SoundCardAttrition/4ch.wav",
    "https://www.mmsp.ece.mcgill.ca/Documents/AudioFormats/WAVE/Samples/Goldwave/addf8-mulaw-GW.wav",
    "https://www.mmsp.ece.mcgill.ca/Documents/AudioFormats/WAVE/Samples/AFsp/M1F1-uint8WE-AFsp.wav",
    "https://www.mmsp.ece.mcgill.ca/Documents/AudioFormats/WAVE/Samples/AFsp/M1F1-mulawWE-AFsp.wav",
    "https://www.mmsp.ece.mcgill.ca/Documents/AudioFormats/WAVE/Samples/SoundCardAttrition/drmapan.wav",
    "http://mmsp.ece.mcgill.ca/Documents/AudioFormats/WAVE/Samples/AFsp/M1F1-int16WE-AFsp.wav"
]

os.makedirs("audios", exist_ok=True)
audio_paths = []

for i, url in enumerate(tqdm(AUDIO_URLS, desc="Downloading audio files")):
    response = requests.get(url)
    if response.status_code == 200:
        path = f"audios/audio_{i}.wav"
        with open(path, "wb") as f:
            f.write(response.content)
        audio_paths.append(path)

In [90]:
import torch
import torchaudio
from transformers import Wav2Vec2Model, Wav2Vec2Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "facebook/wav2vec2-base-960h"

processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name).to(device)


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [91]:
import librosa
import numpy as np

def get_audio_embedding(path):
    waveform, sr = librosa.load(path, sr=16000)
    inputs = processor(waveform, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        hidden_states = outputs.last_hidden_state  # shape: (batch, time, features)
        embedding = hidden_states.mean(dim=1).squeeze().cpu().numpy()  # mean pooling
    return embedding


In [92]:
import faiss

embeddings = []
for path in tqdm(audio_paths, desc="Embedding audios"):
    vec = get_audio_embedding(path)
    embeddings.append(vec)

embeddings = np.stack(embeddings).astype("float32")

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

faiss.write_index(index, "audio_index.faiss")
with open("audio_paths.txt", "w") as f:
    f.writelines([p + "\n" for p in audio_paths])


Embedding audios: 100%|██████████| 6/6 [00:00<00:00, 14.06it/s]


In [93]:
def search_similar_audio(query_path, top_k=6):
    index = faiss.read_index("audio_index.faiss")
    with open("audio_paths.txt", "r") as f:
        paths = [line.strip() for line in f.readlines()]

    query_vec = get_audio_embedding(query_path).astype("float32").reshape(1, -1)
    distances, indices = index.search(query_vec, top_k)

    print(f"Query audio: {query_path}")
    print("Top matches:")
    audioDistance = 0
    audioToShow =""
    for idx, dist in zip(indices[0], distances[0]):
        print(f"{paths[idx]} - Closest Distance: {dist}")
        if(audioDistance == 0):
          audioDistance = {dist}
          audioToShow = {paths[idx]}
        if(audioDistance >= {dist}):
          audioDistance = {dist}
          audioToShow = {paths[idx]}
    print(f"Similar audio file is:{audioToShow}")


In [96]:
search_similar_audio("uploaded/checkFile.wav")


Query audio: uploaded/checkFile.wav
Top matches:
audios/audio_1.wav - Closest Distance: 0.0
audios/audio_0.wav - Closest Distance: 5.936538219451904
audios/audio_2.wav - Closest Distance: 7.421544075012207
audios/audio_3.wav - Closest Distance: 8.535924911499023
audios/audio_5.wav - Closest Distance: 8.587011337280273
audios/audio_4.wav - Closest Distance: 19.593053817749023
Similar audio file is:{'audios/audio_1.wav'}
